# Hour 2 · The genre map — does the corpus sort itself?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/2b_similarity_clustering.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F2b_similarity_clustering.ipynb)


**The headline demo.** We turn every tablet into a vector of its vocabulary, project all of them onto a 2-D map, and colour by genre. The question: **does the machine rediscover the genres philologists already know — without being told them?**

Hover any point to read that tablet. **Needs:** `scikit-learn`, `plotly`, `umap-learn` (all in `requirements.txt`; a PCA fallback runs if UMAP is missing).

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. Turn each tablet into a TF-IDF vector
Each tablet becomes a point in a high-dimensional 'vocabulary space'.

In [ ]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
print(f"{X.shape[0]} tablets in a {X.shape[1]}-word space")

## 2. Squash that space down to 2 dimensions
**UMAP** keeps similar tablets near each other. (If UMAP isn't installed, we fall back to PCA so the cell always runs.)

In [ ]:
import numpy as np
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=12, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X.toarray())
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X.toarray())
    METHOD = f"PCA (UMAP unavailable: {type(e).__name__})"
print("projected with", METHOD)

## 3. The map — hover a point to read the tablet  ⭐
Colours are the *known* genres. Do they land in separate regions on their own?

In [ ]:
import pandas as pd
import plotly.express as px

def preview(t, n=3):
    return "<br>".join(f"{r}: {l}" for r, l in zip(t["refs"][:n], t["lines"][:n]))

df = pd.DataFrame({
    "x": xy[:,0], "y": xy[:,1],
    "KTU": [t["ktu"] for t in sample], "genre": genres,
    "words": [len(t["tokens"]) for t in sample],
    "text": [preview(t) for t in sample]})

fig = px.scatter(df, x="x", y="y", color="genre", hover_name="KTU",
                 hover_data={"x": False, "y": False, "genre": True,
                             "words": True, "text": True},
                 title=f"CUC tablets in vocabulary space ({METHOD}) — hover to read")
fig.update_traces(marker=dict(size=12, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(height=640, legend_title_text="genre",
                  xaxis_title="", yaxis_title="")
fig.show()

## 4. Which tablets are most alike?
The nearest neighbours of one tablet, by shared vocabulary.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
S = pd.DataFrame(cosine_similarity(X), index=labels, columns=labels)
TARGET = "1.4"   # a Baal-cycle tablet
print(f"Closest to KTU {TARGET}:")
print(S[TARGET].drop(TARGET).sort_values(ascending=False).head())

## 5. Let the machine group them blind
`KMeans` clusters with **no** genre labels — then we check the clusters against the genres.

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=4, random_state=0, n_init=10).fit(X)
grid = pd.crosstab(pd.Series(km.labels_, name="cluster"), pd.Series(genres, name="genre"))
grid

## 6. Discussion
- Do letters land together? Do myths separate from rituals? Where does the machine **agree** with philologists, and where does it **disagree** (and is the disagreement interesting)?
- This is the whole workshop in one picture: raw text → vectors → a map that *means* something.

> **Production version:** UgaritLab serves a precomputed UMAP; here you ran a **live, interactive** one over the whole sample — and can reshape it yourself below.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [ ]:
# Re-run sections 1–3 after changing either of these.
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
MY_GENRES = {"myth", "letter", "ritual", "divination", "god-list"}  # ← add/remove a genre
NEIGHBORS = 8        # ← UMAP n_neighbors: smaller = more local clumps, larger = smoother

sample = [t for t in texts if t["genre"] in MY_GENRES and len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
labels, docs = corpus_as_documents(sample)
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=NEIGHBORS, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X.toarray())
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X.toarray()); METHOD = "PCA"
print(f"{len(sample)} tablets — now re-run the map cell (section 3) to see them.")
# Hint: add "god-list" or "epic" and watch whether they carve out their own corner.